# APC-Vis 7B · Full trace study · APC + Orient Anything V2

Inference source: `v0.3.0-ablation.4`, matching the successful sanity setup. Run this notebook in its own Kaggle session with **T4 ×2**, **Internet ON**, and Secret **HF_token** enabled. Attach the corrected **sc203_apc_trace120** Dataset.

**Scope: all 120 diagnostic traces: 40 Left/Right, 40 Visibility, 40 Facing.** `sample_ids=None` selects the whole attached trace cohort. This run covers the selected diagnostic cohort; it does not expand to the complete 3DSRBench dataset.

Branches: **APC + Orient Anything V2**. Baseline predictions are read from the saved APC results. Qwen 7B FP16, Depth Pro maps, viewer selection, and renderer follow the same controlled intervention protocol as sanity.

Run All. Results are saved after each sample. All trace images, prompts, candidate boxes, selections, masks, orientations, scenes and rendered views are retained. Error/invalid answers count as Wrong.

Output: `/kaggle/working/apc_ablation_02_orientanything_v2_v4`. Save/download the whole output folder to preserve the images linked by `index.html`. Environment and weights stay under `/kaggle/temp`. ZIP export is optional to avoid duplicating large outputs.

If interrupted, attach the previous Output and point `resume_input` to that same run folder. Keep `CODE_REF` and the scientific settings unchanged. Completed results are reused; set `retry_errors=True` only to retry failed units.

The two full notebooks can run independently on separate Kaggle accounts with access to the same Input Dataset. Merge their output folders later with `python -m sc203_kat.ablation.merge --inputs OUTPUT_01 OUTPUT_02 --output COMBINED`.


In [ ]:
REPO_URL = 'https://github.com/leetranwyng/sc203-kat.git'
CODE_REF = 'v0.3.0-ablation.4'
HF_SECRET_NAME = 'HF_token'
RUN_CONFIG = {'method': 'apc_vis',
 'mode': 'full',
 'study': '02_orientanything_v2',
 'trace_input': None,
 'model_input': None,
 'resume_input': None,
 'output_dir': '/kaggle/working/apc_ablation_02_orientanything_v2_v4',
 'runtime_dir': '/kaggle/temp/apc_ablation_runtime/02_orientanything_v2',
 'variants': ['oa2'],
 'sample_ids': None,
 'weight_inputs': {'sam3': None, 'oneref': None, 'oneref_tokenizer': None, 'oa2': None},
 'sam3_threshold': 0.5,
 'oneref_min_iou': 0.0,
 'seed': 42,
 'sample_timeout_seconds': 900,
 'phase_timeout_seconds': 21600,
 'disk_min_free_gib': 3.0,
 'retry_errors': False}


In [ ]:
import hashlib, json, os, subprocess, sys
from pathlib import Path

if not CODE_REF or CODE_REF.startswith('-') or any(c.isspace() for c in CODE_REF):
    raise ValueError('CODE_REF must be a Git tag, branch, or full commit.')
if REPO_URL != 'https://github.com/leetranwyng/sc203-kat.git':
    raise ValueError('Use the public leetranwyng/sc203-kat repository.')
GIT_ENV = dict(os.environ, GIT_TERMINAL_PROMPT='0', GCM_INTERACTIVE='Never')
cache = Path(os.environ.get('SC203_CODE_CACHE', '/kaggle/temp/sc203-kat-code'))
cache.mkdir(parents=True, exist_ok=True)
key = hashlib.sha256((REPO_URL + '\n' + CODE_REF).encode()).hexdigest()[:16]
CODE_ROOT = cache / key

def git(*args):
    result = subprocess.run(['git', *args], env=GIT_ENV, text=True,
                            capture_output=True, timeout=300)
    if result.returncode: raise RuntimeError((result.stderr or result.stdout).strip())
    return result.stdout.strip()

if not (CODE_ROOT / '.git').exists():
    if CODE_ROOT.exists() and any(CODE_ROOT.iterdir()):
        raise RuntimeError('Code cache is not empty; choose a different SC203_CODE_CACHE.')
    git('clone', '--no-checkout', '--no-tags', '--depth', '1', REPO_URL, str(CODE_ROOT))
else:
    if git('-C', str(CODE_ROOT), 'remote', 'get-url', 'origin') != REPO_URL:
        raise RuntimeError('Cached source origin differs from REPO_URL.')
    if git('-C', str(CODE_ROOT), 'status', '--porcelain', '--untracked-files=normal'):
        raise RuntimeError('Cached source has local changes; use a new code cache.')
git('-C', str(CODE_ROOT), 'fetch', '--depth', '1', '--no-tags', 'origin', CODE_REF)
commit = git('-C', str(CODE_ROOT), 'rev-parse', '--verify', 'FETCH_HEAD^{commit}')
git('-C', str(CODE_ROOT), 'checkout', '--detach', commit)

ENV = dict(GIT_ENV, SC203_KAT_ROOT=str(CODE_ROOT), PYTHONPATH=str(CODE_ROOT / 'src'))
if HF_SECRET_NAME and not ENV.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(HF_SECRET_NAME)
        if value: ENV['HF_TOKEN'] = value.strip()
        del value
    except Exception:
        print('HF secret unavailable; using public/local model files.')
config_dir = Path(os.environ.get('SC203_CONFIG_DIR', '/kaggle/temp/sc203-kat-configs'))
config_dir.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = config_dir / (RUN_CONFIG['method'] + '_' + RUN_CONFIG['mode'] + '.json')
CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, ensure_ascii=False, indent=2), encoding='utf-8')
prepare = (
    'import json,sys; from pathlib import Path; '
    'from sc203_kat.runtime.kaggle import ensure_submodule,source_metadata; '
    'root=Path(sys.argv[1]); method=sys.argv[2]; '
    'ensure_submodule(root,method); metadata=source_metadata(root,method); '
    'print(json.dumps(metadata,indent=2))'
)
subprocess.run([sys.executable, '-c', prepare, str(CODE_ROOT), RUN_CONFIG['method']], cwd=CODE_ROOT, env=ENV, check=True)
print('Code:', commit, '| Method:', RUN_CONFIG['method'], '| Mode:', RUN_CONFIG['mode'])


In [ ]:
# This orchestration process does not load ML models in the notebook kernel.
subprocess.run([sys.executable, '-u', '-m', 'sc203_kat.ablation.run', '--config', str(CONFIG_PATH)],
               env=ENV, cwd=CODE_ROOT, check=True)


In [ ]:
from IPython.display import FileLink, display
output = Path(RUN_CONFIG['output_dir'])
print('Results:', output)
if (output / 'summary.json').exists():
    import pandas as pd
    summary = json.loads((output / 'summary.json').read_text())
    display(pd.DataFrame(summary['by_task'])[['variant', 'task', 'correct', 'wrong', 'accuracy']])
    display(pd.DataFrame(summary['pairs']))
    for name in ('index.html', 'accuracy.csv', 'paired_outcomes.csv', 'predictions.jsonl', 'summary.json'):
        display(FileLink(str(output / name)))
    print('Download the complete output folder to keep the HTML images and stage evidence.')
else:
    print('Inspect setup output and logs. Completed per-sample results remain in the output folder.')


In [ ]:
# Optional: archive RESULTS only; source code always comes from GitHub.
# Keep false to avoid storing a second copy when Kaggle disk is low.
EXPORT_RESULT_ZIP = False
if EXPORT_RESULT_ZIP:
    import shutil
    if shutil.disk_usage(output).free < 2 * sum(p.stat().st_size for p in output.rglob('*') if p.is_file()):
        raise RuntimeError('Not enough disk to duplicate evidence as ZIP; download the output folder instead.')
    archive = shutil.make_archive(str(output), 'zip', root_dir=output.parent, base_dir=output.name)
    display(FileLink(archive))
